# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library, following the FAIR principles for AI-ready data.

### Dataset Source
The dataset is described by a Croissant schema accessible at:
<br>
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

---

In [ ]:
# Ensure `mlcroissant` is available in this environment
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and prepare the dataset for exploration using `mlcroissant`. This will download the schema, validate, and give access to the record sets defined in this data package.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset title:", metadata.name)
print("Description:", metadata.description)

# Optionally show basic info
print(f"Publication date: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Keywords: {metadata.keywords}")


## 2. Data Overview
Listing available record sets, fields, and their `@id`s, as referenced by the Croissant schema.

> All entities are referenced by their `@id` for precise targeting and reproducibility.

In [ ]:
# Inspect record sets available in the dataset
record_sets = list(dataset.record_sets())

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet '@id': {rs['@id']}")
    print(f"  name: {rs.get('name','')}\n  description: {rs.get('description','')}\n  fields:")
    for fld in rs.get('field', []):
        if hasattr(fld, '__getitem__') and '@id' in fld:
            print(f"    - {fld['@id']}")
        elif isinstance(fld, str):
            print(f"    - {fld}")
    print("-")

## 3. Data Extraction
Load data from selected record sets using their `@id`. Records are loaded into pandas DataFrames for convenient downstream analysis.

> Here, we extract all main record sets. Replace the list or elements below to select desired record sets for your workflows.

In [ ]:
# List of record set @ids (update this cell after printing record sets above if you want to change them)
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

# Extract and load records as DataFrames, keyed by record set @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet '@id': {record_set_id} shape={df.shape}")

# Preview one main DataFrame (replace index if multiple record sets)
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"\nColumns in '{preview_id}':\n", dataframes[preview_id].columns.tolist())
    dataframes[preview_id].head()


## 4. Exploratory Data Analysis (EDA)
Clean and explore data:

- Filter records with a threshold on a numeric field (e.g. PHQ-9 score, GAD-7 score, etc.)
- Normalize the numeric field
- Optionally group by a categorical field

> *Replace* the field `@id`s below for other analyses, as needed. Note: field names correspond to `@id`s, which usually match the CSV column headings.

In [ ]:
# Choose a record set and numeric field for EDA (adjust to your schema)
record_set_id = preview_id  # Example: first record set
df = dataframes[record_set_id]

# List all column names
print("Columns:", df.columns.tolist())

# Example: choose a common numeric field (adjust to your needs)
numeric_field = None
candidate_fields = [
    field for field in df.columns if df[field].dtype in [int, float, 'int64', 'float64'] or (df[field].dtype == object and df[field].str.replace('.','').str.isdigit().any())
]
if candidate_fields:
    numeric_field = candidate_fields[0]

if numeric_field:
    # Attempt to convert numeric if needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records.")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field (categorical)
    group_fields = [c for c in df.columns if c != numeric_field]
    group_field = None
    for c in group_fields:
        if df[c].nunique() < 10 and pd.api.types.is_string_dtype(df[c]):
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped (mean) {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Here we plot the distribution of the selected numeric field, and (if applicable) show comparisons by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using its Croissant metadata and `mlcroissant`
- Explored the metadata and record set structure, referring to all elements by their `@id`
- Loaded data into pandas and performed basic EDA including filtering and normalization
- Visualized simple distributions and group differences

This provides a foundation for deeper analytics using the dataset. For production or research settings, always review the detailed schema and refer to [mlcroissant documentation](https://mlcommons.org/croissant/) for advanced usage.

> **Remember:** This analysis references fields, record sets, and columns by their Croissant schema `@id` for full reproducibility and FAIRness.
